# 🧠 Fine-Tuning Real: Gemma 2 2B + Apex AI

Este notebook faz fine-tuning **real** do Gemma 2 2B com seus dados.
Após o treino, você baixa o modelo e usa localmente com **Ollama** — SEM depender de API nenhuma.

## ⚡ Requer GPU gratuita (T4) — vai em Runtime → Change runtime type → T4 GPU

In [7]:
# 1. Instalar dependencias
!pip install -q accelerate peft transformers trl bitsandbytes datasets sentencepiece 2>&1 | tail -1
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


In [8]:
import json, os, torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print('✅ Bibliotecas carregadas')

✅ Bibliotecas carregadas


---
## 📤 PASSO 1: Enviar seu arquivo de treino

**Execute a celula abaixo e faça upload do arquivo `apex_training_vertex.jsonl`**

> Se você nao tem o arquivo, ele será baixado automaticamente do GitHub.

In [9]:
from google.colab import files
import urllib.request

# Tenta baixar do GitHub primeiro
url = "https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data/apex_training_vertex.jsonl"
local_file = "apex_training_vertex.jsonl"

try:
    urllib.request.urlretrieve(url, local_file)
    print(f'✅ Baixado do GitHub: {local_file}')
except Exception as e:
    print(f'⚠️ GitHub indisponivel: {e}')
    print('➡️  Faça upload manual do arquivo .jsonl')
    uploaded = files.upload()
    local_file = list(uploaded.keys())[0]
    print(f'✅ Upload feito: {local_file}')

✅ Baixado do GitHub: apex_training_vertex.jsonl


In [10]:
# 2. Carregar e preparar o dataset
raw_data = []
with open(local_file, 'r') as f:
    for line in f:
        raw_data.append(json.loads(line))

print(f'📊 Total de exemplos: {len(raw_data)}')
print('Primeiro exemplo:')
print(json.dumps(raw_data[0], indent=2)[:500])

📊 Total de exemplos: 845
Primeiro exemplo:
{
  "systemInstruction": "Voc\u00ea \u00e9 a Apex AI, uma plataforma de assist\u00eancia profissional para arquitetura, constru\u00e7\u00e3o, BIM, marketing e gest\u00e3o. Responda sempre em portugu\u00eas (ou no idioma do usu\u00e1rio) de forma t\u00e9cnica, direta e \u00fatil. Nunca invente dados, integra\u00e7\u00f5es ou aprova\u00e7\u00f5es que n\u00e3o existem \u2014 seja honesta sobre o que a plataforma faz de verdade.",
  "contents": [
    {
      "role": "user",
      "parts": [
        


In [11]:
# 3. Formatar no padrao instruction-tuning
def format_example(example):
    if 'messages' in example:
        parts = example['messages']
        user = next((m['content'] for m in parts if m['role']=='user'), '')
        assistant = next((m['content'] for m in parts if m['role']=='assistant'), '')
    elif 'prompt' in example:
        user = example['prompt']
        assistant = example.get('completion', example.get('response', ''))
    else:
        user = str(example.get('input', example.get('instruction', '')))
        assistant = str(example.get('output', example.get('response', '')))

    return {
        'text': f"### Instrucao:\n{user}\n\n### Resposta:\n{assistant}"
    }

formatted = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f'✅ Dataset pronto: {len(dataset["train"])} treino, {len(dataset["test"])} teste')

✅ Dataset pronto: 760 treino, 85 teste


In [17]:
# 4. Carregar modelo Gemma 2 2B em 4-bit
from huggingface_hub import notebook_login

notebook_login()

model_name = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)
print('✅ Modelo carregado em 4-bit')

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
403 Client Error. (Request ID: Root=1-6a49cd9a-0a3ae9b8465cf4ee597a1f29;4537e976-62a0-431d-936e-72a901360e1e)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2-2b-it to ask for access.

In [14]:
# 5. Configurar LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Deve mostrar ~0.5% de parametros treinaveis

NameError: name 'model' is not defined

In [18]:
# 6. TREINAMENTO REAL 🔥
training_args = TrainingArguments(
    output_dir="./gemma-apex-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    max_seq_length=512,
    dataset_text_field="text",
)

print('🚀 Iniciando treinamento...')
trainer.train()
print('✅ Treinamento concluido!')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


NameError: name 'model' is not defined

In [20]:
# 7. Salvar modelo + tokenizer
save_path = "./gemma-apex-final"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f'✅ Modelo salvo em {save_path}')

NameError: name 'trainer' is not defined

In [21]:
# 8. Compactar para download
import shutil
shutil.make_archive('gemma-apex-finetuned', 'zip', save_path)
filesize = os.path.getsize('gemma-apex-finetuned.zip') / 1024 / 1024
print(f'📦 Arquivo compactado: gemma-apex-finetuned.zip ({filesize:.1f} MB)')
print('\n⬇️  Faça o download clicando no link abaixo:')

FileNotFoundError: [Errno 2] No such file or directory: './gemma-apex-final'

In [22]:
from google.colab import files
files.download('gemma-apex-finetuned.zip')

FileNotFoundError: Cannot find file: gemma-apex-finetuned.zip

---
## 🎯 Como usar o modelo treinado

### Opção 1: Local com Ollama (recomendado)

```bash
# No seu computador:
ollama pull gemma:2b
ollama create apex-ai-trained --file Modelfile.apex
ollama run apex-ai-trained
```

### Opção 2: Python (Hugging Face)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('./gemma-apex-final')
tokenizer = AutoTokenizer.from_pretrained('./gemma-apex-final')
```

---
## ✅ Treino REAL concluído!
Agora a Apex AI roda **sem depender de API nenhuma** - apenas seu modelo local.